# 05 — Reasoning Effort

**Let the model think harder — with one provider-agnostic knob.**

Modern LLMs can spend extra tokens *thinking* before they answer — but every
provider exposes that differently. Anthropic takes a thinking-token budget,
OpenAI takes a categorical effort, Google takes a thinking config, OpenRouter
takes a reasoning object. Orqest collapses all four into a single
`BaseAgent(reasoning=...)` knob. This notebook —

1. translates one effort level into all four provider shapes (`resolve_reasoning_settings`),
2. proves the knob threads through to the model (`agent.agent.model_settings`),
3. runs an honest A/B on a hard problem at `low` vs `high` effort,
4. spends the budget where it matters — a `Pipeline` with a cheap step and a thinking step.

**Prerequisites:** a `.env` at the repo root with `LLM_API_KEY` and `LLM_MODEL` (see the main README).

## 1. Load config

In [1]:
from orqest import load_config

config = load_config()
print(f"Model: {config.llm_model}")

Model: openai:gpt-5.2


## 2. One knob, every provider

pydantic-ai 1.x has no unified thinking API — each provider's model reads its
own `ModelSettings` key. `resolve_reasoning_settings` is the single place that
knows the four shapes: you hand it a provider and one effort level, it hands
back the provider-specific settings. (This is pure translation — no LLM calls.)

In [2]:
from orqest.utils.reasoning import resolve_reasoning_settings

for provider in ["openai", "anthropic", "google", "openrouter"]:
    print(f"{provider:11s} -> {resolve_reasoning_settings(provider, 'high')}")

openai      -> {'openai_reasoning_effort': 'high'}
anthropic   -> {'anthropic_thinking': {'type': 'enabled', 'budget_tokens': 24576}, 'max_tokens': 32768}
google      -> {'google_thinking_config': {'thinking_budget': 24576, 'include_thoughts': True}, 'max_tokens': 32768}
openrouter  -> {'openrouter_reasoning': {'effort': 'high', 'enabled': True}}


One vocabulary in, four shapes out:

- **openai** gets a categorical `openai_reasoning_effort` — passed straight through.
- **anthropic** and **google** get a *token budget* derived from the effort — plus a
  `max_tokens` headroom fill, because Anthropic *requires* `max_tokens > budget_tokens`
  or the call fails. Orqest fills it so reasoning works out of the box.
- **openrouter** gets its own `openrouter_reasoning` object.

Everything downstream — `BaseAgent`, the orchestration primitives — only ever
sees `model_settings`. The provider divergence stops here.

## 3. The knob takes effect

`reasoning` is a `BaseAgent` constructor kwarg. It is translated against the
agent's *own* provider and merged into `model_settings` — and explicit
`model_settings` keys win on conflict. You don't need to call the model to see
it land: just inspect `agent.agent.model_settings`.

In [3]:
from pydantic import BaseModel, Field
from pydantic_ai.settings import ModelSettings
from orqest.agents import BaseAgent, GlobalState


class Solution(BaseModel):
    """A worked answer: the method, the steps, the final number."""
    approach: str = Field(description="One sentence: the method used.")
    steps: list[str] = Field(description="The working, one concise step per item.")
    final_answer: int = Field(description="The final numeric answer.")


class SolverAgent(BaseAgent[GlobalState, Solution]):
    async def _run_implementation(self, state: GlobalState, **kwargs) -> Solution:
        result = await self.call_model(state.get_latest_message("user"), state)
        return result.output


def build_solver(reasoning=None, model_settings=None) -> SolverAgent:
    """A SolverAgent on the configured model, with an optional reasoning effort."""
    return SolverAgent(
        agent_name="solver",
        system_prompt="You are a careful mathematician. Show each step.",
        output_type=Solution,
        model=config.llm_model,
        api_key=config.llm_api_key,
        reasoning=reasoning,
        model_settings=model_settings,
    )


plain = build_solver()
thinking = build_solver(reasoning="high")
mixed = build_solver(reasoning="high", model_settings=ModelSettings(temperature=0.2))

print("reasoning=None        ->", plain.agent.model_settings)
print("reasoning='high'      ->", thinking.agent.model_settings)
print("'high' + temperature  ->", mixed.agent.model_settings)

reasoning=None        -> None
reasoning='high'      -> {'openai_reasoning_effort': 'high'}
'high' + temperature  -> {'openai_reasoning_effort': 'high', 'temperature': 0.2}


`reasoning=None` leaves the provider defaults untouched. `reasoning="high"`
adds the provider-specific key. The mixed case shows the merge: both the
reasoning key *and* the explicit `temperature` survive.

Our configured model is OpenAI, so we see `openai_reasoning_effort` — the
*same* `reasoning="high"` against an `anthropic:...` model would have produced
`anthropic_thinking` instead (section 2). The agent code never changes; only
the model string does.

## 4. An honest A/B on a hard problem

Now actually run it. The problem: count the distinct ways to make exactly one
dollar from pennies, nickels, dimes, and quarters — a genuine combinatorial
counting problem (the correct answer is **242**). We run the same agent twice,
at `low` and `high` effort, and time both.

In [4]:
import time

PROBLEM = (
    "How many distinct ways can you make exactly one US dollar (100 cents) "
    "using any number of pennies (1c), nickels (5c), dimes (10c), and "
    "quarters (25c)? Order does not matter."
)


async def solve(effort):
    agent = build_solver(reasoning=effort)
    state = GlobalState()
    state.add_message("user", PROBLEM)
    started = time.perf_counter()
    out = await agent.run(state)
    elapsed = time.perf_counter() - started
    mark = "correct" if out.final_answer == 242 else "WRONG"
    print(f"[{effort:>4}]  answer={out.final_answer} ({mark})  {elapsed:4.1f}s")
    print(f"         approach: {out.approach}")
    return out, elapsed


low_out, low_t = await solve("low")
high_out, high_t = await solve("high")

[ low]  answer=242 (correct)  12.6s
         approach: Reduce by 5 cents and sum over allowable numbers of quarters and dimes; pennies are then forced by the remainder.


[high]  answer=242 (correct)  14.1s
         approach: Casework on number of quarters; for each case count (dimes,nickels) pairs since pennies are forced.


**The honest read.** Both efforts land on 242 — `gpt-5.2` is strong enough that
its *floor* already clears this problem. What `high` reliably buys is more
deliberation, which you pay for in **latency** (and tokens) — not a correctness
jump on every problem.

So `reasoning` is a *dial*, not a guarantee. Spend it where the problem sits at
the edge of the model's ability, and **measure** — don't assume a higher number
is always better. (Same dogfooding honesty as notebook 01's confidence lesson:
the feature works; the LLM's behaviour is its own thing.)

## 5. Spend the budget where it matters

`reasoning` is set *per agent* — so in a multi-step workflow you give the cheap
steps no budget and the hard step a high one. Here a `Pipeline` does exactly
that: a `reasoning=None` triage step restates the problem, then a
`reasoning="high"` solver does the real work.

Steps auto-coerce — a `BaseAgent` becomes an `AgentStep`, a plain callable
becomes a `FunctionStep` — so a thin adapter bridges the triage agent's output
into the solver's prompt.

In [5]:
from orqest.orchestration import Pipeline


class Restatement(BaseModel):
    """A problem, cleaned up and made explicit."""
    restated: str = Field(
        description="The problem restated precisely, with every quantity named."
    )


class TriageAgent(BaseAgent[GlobalState, Restatement]):
    async def _run_implementation(self, state: GlobalState, **kwargs) -> Restatement:
        result = await self.call_model(state.get_latest_message("user"), state)
        return result.output


# Cheap step: no reasoning budget — restating a problem does not need one.
triage = TriageAgent(
    agent_name="triage",
    system_prompt="Restate the user's problem precisely. Name every quantity. Do not solve it.",
    output_type=Restatement,
    model=config.llm_model,
    api_key=config.llm_api_key,
)

# Expensive step: high reasoning budget — this is where the thinking matters.
solver = build_solver(reasoning="high")


async def to_prompt(r: Restatement) -> str:
    """Adapter: bridge the triage output into the solver's prompt."""
    return f"Solve this problem, showing each step:\n\n{r.restated}"


pipeline = Pipeline([triage, to_prompt, solver], name="triage_then_solve")

WORD_PROBLEM = (
    "A bookstore sells novels for $12 and comics for $5. On Monday it sold "
    "17 items in total for $148. How many novels did it sell?"
)

result = await pipeline.run(WORD_PROBLEM)
print(f"approach: {result.approach}")
for i, step in enumerate(result.steps, 1):
    print(f"  {i}. {step}")
print(f"final_answer (novels): {result.final_answer}")

approach: Solve the system by substitution.
  1. From n + c = 17, solve for c: c = 17 − n.
  2. Substitute into 12n + 5c = 148: 12n + 5(17 − n) = 148.
  3. Distribute: 12n + 85 − 5n = 148.
  4. Combine like terms: 7n + 85 = 148.
  5. Subtract 85 from both sides: 7n = 63.
  6. Divide by 7: n = 9.
final_answer (novels): 9


The triage step ran on the provider defaults — no extra thinking budget. The
solver ran with `reasoning="high"`. Because `reasoning` lives on the *agent*, it
flows through orchestration untouched: `Pipeline` (and `Parallel`, `Router`)
never need to know it exists. You set it once, where the work is hard.

## What's happening under the hood

- `reasoning` is translated by `resolve_reasoning_settings` against the agent's
  provider — taken from a `provider:model_id` string, or from a `Model`
  instance's `.system`. The result is merged into `model_settings`, with
  explicit keys winning.
- pydantic-ai 1.x has no unified thinking API — each provider model reads its
  own `ModelSettings` key. `resolve_reasoning_settings` is the *one* place that
  knows the four shapes; everything downstream just sees `model_settings`.
- For the budget-based providers (Anthropic, Google) orqest fills a `max_tokens`
  headroom when you have not set one — Anthropic *requires*
  `max_tokens > budget_tokens`, so reasoning works out of the box.
- **It translates; it does not validate.** `reasoning` does not check that the
  model can actually reason. A sharp edge worth knowing: this notebook's
  `gpt-5.2` accepts `low` / `medium` / `high` but **rejects `minimal`** with a
  400 — the effort vocabulary is the union across providers, and not every
  model supports every level. Pair `reasoning` with a model you know is
  reasoning-capable, at an effort it supports.
- When pydantic-ai ships its unified `Thinking` capability, orqest's
  `ReasoningEffort` maps onto it 1:1 and this translation layer can be deleted —
  the agent-facing `reasoning="high"` knob stays exactly the same.